## Preliminares

In [1]:
import pandas as pd
import numpy as np
from src.config import data_folder
%load_ext autoreload
%autoreload 2
from src.transform import *

In [2]:
# Abrir archivo raw_data
df = pd.read_parquet(f"{data_folder}/raw_data.parquet")

# Se asegura el ordenamiento por fecha
df = df.sort_values(by='Date').reset_index(drop=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11389 entries, 0 to 11388
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11389 non-null  datetime64[ns]
 1   Ticker                       11389 non-null  object        
 2   Close                        11389 non-null  float64       
 3   Open                         11389 non-null  float64       
 4   Volume                       11389 non-null  float64       
 5   DateAdded                    6720 non-null   object        
 6   Sector                       11367 non-null  object        
 7   Industry                     11367 non-null  object        
 8   TotalRevenue                 11389 non-null  float64       
 9   GrossProfit                  10536 non-null  float64       
 10  OperatingIncome              11389 non-null  float64       
 11  NetIncome                    11389 non-nu

* Para mejorar la visualización de los datos, se expresan las columnas financieras y volumen en millones:

In [3]:
df = columnas_en_millones(df)

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11389,2023-08-25 14:51:30.771797248,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11389.0,1039.58324,0.57,41.207058,82.970001,165.788284,775000.0,22848.031796
Open,11389.0,1007.103576,0.53,40.572777,81.792908,161.548668,775648.0,22098.066663
Volume,11389.0,330.710827,0.0,44.616429,103.7176,259.9822,36280.458,1254.720618
TotalRevenue,11389.0,6628.100052,-12134.0,1277.0,2388.0,5298.872,227173.0,15005.216889
GrossProfit,10536.0,2376.388472,-24353.0,423.0,833.25,1832.09325,137192.0,6412.054808
OperatingIncome,11389.0,974.133078,-55512.0,125.8,304.146,825.0,77648.0,2985.238718
NetIncome,11389.0,726.598954,-43621.0,65.923,199.731,583.0,62578.0,2722.226007
EBITDA,1892.0,1625.434914,-6187.0,200.4785,473.415,1238.2,84427.0,5431.316062
BasicAverageShares,11342.0,645.376445,0.00021,96.045056,220.0,551.7325,30864.0,1758.57522


## Corrección de errores y valores perdidos

In [5]:
# Se analiza el siguiente registro con valores extremadamente elevados en el estado de resultados
registro_iiin = df[(df['Ticker'] == 'IIIN') & (df['Date'] == '2021-12-01')]

# Mostrar el resultado
print(registro_iiin.T)

                                            2498
Date                         2021-12-01 00:00:00
Ticker                                      IIIN
Close                                  28.253126
Open                                   31.382212
Volume                                    7.3179
DateAdded                                   None
Sector                               Industrials
Industry                       Metal Fabrication
TotalRevenue                             171.258
GrossProfit                               39.919
OperatingIncome                           32.598
NetIncome                                 25.152
EBITDA                                       NaN
BasicAverageShares                        19.344
CashAndCashEquivalents                   89884.0
CurrentDebt                                  NaN
LongTermDebt                                 NaN
TotalDebt                                    NaN
StockholdersEquity                      302038.0
TotalAssets         

* Se observa una desconexión en los valores del Balance General, estan multiplicados por 1.000.
* El caso se detecto ya que arrojaba valores extremos en las métricas.

Se corrige dividiendo las columnas afectadas por 1.000:

In [6]:
condicion_error_balance = (df['Ticker'] == 'IIIN') & (df['Date'] == '2021-12-01')

# Columnas del balance que tienen el error
columnas_error_balance = [
    'CashAndCashEquivalents', 
    'TotalAssets', 
    'StockholdersEquity', 
    'CurrentAssets', 
    'CurrentLiabilities'
]

# Se dividen por 1000
df.loc[condicion_error_balance, columnas_error_balance] /= 1000

Anomalías de signo: 

Las siguientes variables no pueden ser negativas:
* `TotalRevenue`
* `CurrentDebt`
* `LongTermDebt`
* `DepreciationAndAmortization`

In [7]:
# Visualizar casos de TotalRevenue negativo
condicion_revenue_negativo = df['TotalRevenue'] < 0
cols_a_visualizar_revenue = ['Ticker', 'Date', 'TotalRevenue', 'OperatingIncome']
anomalias = df.loc[condicion_revenue_negativo, cols_a_visualizar_revenue]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

     Ticker       Date  TotalRevenue  OperatingIncome
587    AVNT 2021-03-01     -30.20000       -12.700000
754     ALB 2021-03-01   -1790.31392      -114.758843
791     XPO 2021-03-01   -1093.00000        14.000000
2099    EMR 2021-12-01    -357.00000      -206.000000
2595    XPO 2022-03-01   -2243.00000      -189.000000
2673   FTNT 2022-03-01   -1415.00000       213.700000
2919    BHF 2022-03-01    -150.00000      2266.000000
4110      J 2022-12-01   -1258.70300       -69.426000
4518    BAX 2023-03-01    -704.00000      -285.000000
4642    BHF 2023-03-01    -127.00000        40.000000
4835     GE 2023-03-01  -12134.00000      4010.000000
4950    CNA 2023-03-01   -3542.00000     -4648.000000
5055    NGL 2023-06-01    -967.05000        65.932000
5523    WDC 2023-09-01   -3391.00000        93.000000
6186   OTIS 2023-12-01   -3543.00000      -516.000000
6345      J 2023-12-01   -1212.28200      -120.924000
6482   NDAQ 2024-03-01    -522.00000       481.000000
6592    FTV 2024-03-01    -5

* No se puede concluir que se traten de alteraciones de signo en el parseo de datos de simFin. Se observa en 2 casos que el valor absoluto de `TotalRevenue` es mayor que `OperatingIncome`, es imposible que los resultados sean mayores que las ventas.

Se opta por asignar todos estos valores anómalos a NaN.

In [8]:
df['TotalRevenue'] = np.where(condicion_revenue_negativo, np.nan, df['TotalRevenue'])

* Casos de deuda negativa: se calcula "PasivoImplicito", que surge de aplicar la ecuación contable fundamental `Activo` = `Pasivo` + `Patrimonio Neto`

In [9]:
# Calcular pasivo implícito
df['PasivoImplicito'] = df['TotalAssets'] - df['StockholdersEquity']

condicion_deuda_negativa = (df['CurrentDebt'] < 0) | (df['LongTermDebt'] < 0)
cols_a_visualizar_deuda = ['Ticker', 'Date', 'TotalDebt', 'CurrentDebt', 'LongTermDebt', 'PasivoImplicito']
anomalias = df.loc[condicion_deuda_negativa, cols_a_visualizar_deuda]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

     Ticker       Date  TotalDebt  CurrentDebt  LongTermDebt  PasivoImplicito
3948    IEP 2022-09-01        NaN     -746.000      7134.000        15991.000
4027    IEP 2022-12-01        NaN     -745.000      7127.000        16908.000
6494   STLD 2024-03-01        NaN      459.987     -3286.537          171.287
7445   STLD 2024-06-01        NaN      425.696     -3570.028         -203.283
8422   STLD 2024-12-01        NaN      882.013     -3997.348         -218.698
8592    TXN 2025-03-01        NaN      750.000    -28049.000       -22289.000
Cantidad de casos: 6


* Se observa que la ecuación contable fundamental no se cumple. 

Se decide eliminar estos registros "tóxicos" del dataset.

In [10]:
df = df[~condicion_deuda_negativa].reset_index(drop=True)
df = df.drop(columns= 'PasivoImplicito')

* Casos de negativos en `DepreciationAndAmortization`: 

se analizan los casos considerando la ecuación `EBITDA` = `OperatingIncome` + `DepreciationAndAmortization`

In [13]:
condicion_depre_negativa = df['DepreciationAndAmortization'] < 0
cols_a_visualizar_depre = ['Ticker', 'Date', 'DepreciationAndAmortization', 'EBITDA', 'OperatingIncome', 'FinancialsSource']
anomalias = df.loc[condicion_depre_negativa, cols_a_visualizar_depre]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

      Ticker       Date  DepreciationAndAmortization  EBITDA  OperatingIncome  \
0         HD 2020-09-01                     -519.000     NaN         6067.000   
3       BURL 2020-09-01                      -54.404     NaN          -81.224   
6        TGT 2020-09-01                     -542.000     NaN         2300.000   
7        ADI 2020-09-01                     -107.077     NaN          450.954   
8         DY 2020-09-01                      -44.129     NaN           54.482   
...      ...        ...                          ...     ...              ...   
9720    ORCL 2025-09-01                     -544.000     NaN         5191.000   
9813     HPE 2025-09-01                      -37.000     NaN          294.000   
9819    CIEN 2025-09-01                       -6.545     NaN           34.790   
9839    LULU 2025-09-01                       -1.630     NaN          438.625   
11279    GME 2026-06-01                       -0.300   144.9          144.900   

      FinancialsSource  
0 

* Dato de `yfinance`: En este caso se observa que el valor implicito de `DepreciationAndAmortization` es igual a cero (EBITDA = OperatingIncome).
El valor de -0.3 pudo haberse tratado de un pequeño "ajuste contable", $300.000 dólares para una compañia del tamaño de `GameStop` es contablemente irrelevante. Se reemplaza dicho valor por cero.

* Datos de `simFin`: utilizan signo negativo para esta columna, lo cual es incorrecto. Dichos valores serán convertidos a positivos.

In [ ]:
# Se reemplaza por 0 el valor de yfinance
condicion_depre_negativa_yfinance = (df['DepreciationAndAmortization'] < 0) & (df['FinancialsSource']=='yfinance')
df.loc[condicion_depre_negativa_yfinance,'DepreciationAndAmortization'] = 0

# Se convierten a positivos los de simFin
condicion_depre_negativa_simfin = (df['DepreciationAndAmortization'] < 0) & (df['FinancialsSource']=='simFin')
df.loc[condicion_depre_negativa_simfin, 'DepreciationAndAmortization'] = df.loc[condicion_depre_negativa_simfin, 'DepreciationAndAmortization'].abs()

# Verificar cambios
anomalias = df.loc[condicion_depre_negativa, cols_a_visualizar_depre]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

      Ticker       Date  DepreciationAndAmortization  EBITDA  OperatingIncome  \
0         HD 2020-09-01                      519.000     NaN         6067.000   
3       BURL 2020-09-01                       54.404     NaN          -81.224   
6        TGT 2020-09-01                      542.000     NaN         2300.000   
7        ADI 2020-09-01                      107.077     NaN          450.954   
8         DY 2020-09-01                       44.129     NaN           54.482   
...      ...        ...                          ...     ...              ...   
9720    ORCL 2025-09-01                      544.000     NaN         5191.000   
9813     HPE 2025-09-01                       37.000     NaN          294.000   
9819    CIEN 2025-09-01                        6.545     NaN           34.790   
9839    LULU 2025-09-01                        1.630     NaN          438.625   
11279    GME 2026-06-01                        0.000   144.9          144.900   

      FinancialsSource  
0 

In [ ]:
df.describe().T

In [ ]:
# Incidencia de missings
mostrar_missings(df)

* Se imputan parte de los NaNs en Variables Financieras antes de calcular métricas, 
mediante las relaciones contables entre ellas:

In [ ]:
df_acc_imputed = imputar_equivalencias_financieras(df)
mostrar_missings(df_acc_imputed)

* Se crea feature `YearsSinceAdded`:

In [ ]:
#  Pasar DateAdded a formato datetime, los NaN se vuelven NaT (not a time)
df_acc_imputed['DateAdded'] = pd.to_datetime(df_acc_imputed['DateAdded'], errors='coerce')
# Convertir a YearsSinceAdded, aqui los nulos regresan a NaN
df_acc_imputed['YearsSinceAdded'] = round(((pd.Timestamp.now() - df_acc_imputed['DateAdded']).dt.days / 365.25), 0)

# 3. Se asignan a cero años los tickers que no pertenecen al Índice S&P 500
df_acc_imputed['YearsSinceAdded'] = df_acc_imputed['YearsSinceAdded'].fillna(0)

# Eliminar la columna original
df_acc_imputed.drop('DateAdded', axis=1, inplace=True)
mostrar_missings(df_acc_imputed)

In [ ]:
# Se imputan las columnas financieras originales, por su media o mediana móvil según sus asimetrías
df_fin_imputed = imputar_numericas(df_acc_imputed)
mostrar_missings(df_fin_imputed)

* Se aplica forward fill y back fill para cubrir posibles huecos (es necesario que no hayan valores perdidos antes de calcular los valores TTM):

In [ ]:
df_fin_imputed = aplicar_fill(df_fin_imputed, limite=None)
mostrar_missings(df_fin_imputed)

* Se convierten las variables de flujo trimestrales a valores TTM (ventana móvil de 4 trimestres):

In [ ]:
df_ttm = transformar_flujos_a_ttm(df_fin_imputed)
mostrar_missings(df_ttm)

* Calcular métricas:

In [ ]:
df_with_metrics, crecimiento_cols = calcular_metricas(df_ttm)
mostrar_missings(df_with_metrics)

* Se aplica imputación transversal para las columnas de crecimiento:

In [ ]:
df_with_metrics = imputar_transversal(df_with_metrics, crecimiento_cols)
mostrar_missings(df_with_metrics)

* Calcular los retornos trimestrales, varianza del activo y covarianza con el mercado para cada ticker:

In [ ]:
# Se abre el fichero de precios del Índice del Mercado
df_index = pd.read_parquet(f"{data_folder}/market_index.parquet")
df_with_features = calcular_retornos(df_with_metrics, df_index)
mostrar_missings(df_with_features)

In [ ]:
df_with_features.describe().T

In [ ]:
# Analizar extremos
df_min = df_with_features.loc[df_with_features['EnterpriseValue'].idxmin()]
df_min

In [ ]:
# Se aplica lag de 1 período a volumen para evitar Data Leakage
df_with_features['Volume_Lag1'] = df_with_features['Volume'].shift(1)
df_with_features.drop('Volume', axis=1, inplace=True)

Imputación final de Valores Perdidos

In [ ]:
# Se aplica la imputación numérica sobre las nuevas variables
df_imputed = imputar_numericas(df_with_features)
mostrar_missings(df_imputed)

In [ ]:
# Se aplican los fills sobre los missings que queden
df_imputed = aplicar_fill(df_imputed,None)
mostrar_missings(df_imputed)

## Transformaciones

In [ ]:
# Se calculan tamaños relativos: RelativeAssets y RelativeRevenue
df_transformed = calcular_relative_size(df_imputed)
df_transformed.info()

In [ ]:
# Convertir Sector y Industry a tipo category
df_transformed['Sector'] = df_transformed['Sector'].astype('category')
df_transformed['Industry'] = df_transformed['Industry'].astype('category')

# Valores unicos en Sector
df_transformed['Sector'].value_counts()

In [ ]:
# Valores unicos en Industry
df_transformed['Industry'].value_counts()

In [ ]:
# Distribución de variables contínuas
df_transformed.describe().round(4).T

In [ ]:
# Coeficientes de asimetría
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Graficar
columna_a_graficar = 'EnterpriseValue' # indicar columna para el grafico
plot(df_transformed[columna_a_graficar])

In [ ]:
# Transformaciones logarítmicas

columnas_a_transformar = [ 
    'Volume_Lag1',
    'CapExToRevenue',
    'DebtToEquity',
    'QuarterlyVariance_Lag1',
    'MarketCap',
    'EnterpriseValue'
    ]
for columna in columnas_a_transformar:
    df_transformed[columna] = df_transformed[columna].fillna(0)
    df_transformed[f'{columna}_Log1p'] = np.log1p(df_transformed[columna])
    df_transformed.drop(columna, axis=1, inplace=True)

# Coeficientes de asimetria actualizado
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Definir columnas que saltean la "winsorización"
cols_fin_clean = obtener_cols_financieras(incluirTTM=True)

columnas_intactas = cols_fin_clean + [
    # Variables de precio y ratios
    'Close',
    'Open',    
    'TrailingPE',
    'EnterpriseToEbitda',
    'PriceToBook',
    # Otras
    'Date', 
    'Ticker'   
    ]

# Separar el dataset
df_passthrough = df_transformed[columnas_intactas].copy()
df_transformed_features = df_transformed.drop(columns=columnas_intactas)

## Gestión de Outliers

Se winsorizan los valores atipicos en las variables continuas que cumplan los siguientes criterios:

Para variables simetricas:
* A mas de 3 desviaciones tipicas de la media.
* Mas de 3 rangos intercuartilicos.

Para variables asimetricas (modulo del coeficiente de asimetrica mayor a 1):
* A mas de 3 MADs de la mediana.
* Mas de 3 rangos intercuartilicos.

In [ ]:
# Outliers
df_cont_transformed = df_transformed_features.select_dtypes(include="number")
df_winsor = df_cont_transformed.apply(lambda x: gestiona_outliers(x, clas='winsor'))

In [ ]:
# Coeficientes de asimetria luego de winsorizar
df_winsor.skew().sort_values(ascending=False)

In [ ]:
# Visualizar cambios
columna_a_graficar = 'MarketCap_Log1p' # indicar columna para el grafico
plot(df_winsor[columna_a_graficar])

In [ ]:
df_winsor.describe().T

## Concatenación final de columnas

In [ ]:
df_non_numeric_transformed = df_transformed_features.select_dtypes(exclude='number')
# Se unen variables contínuas transformadas y variables no numéricas
df_combined = pd.concat([df_non_numeric_transformed, df_winsor], axis=1)

# Unir con las columnas que fueron salteadas
df_final = pd.concat([df_passthrough, df_combined], axis=1)
df_final.info()

In [ ]:
# Guardar datos extraidos en fichero clean_data
df_final.to_parquet(f"{data_folder}/clean_data.parquet")
print(f"Fichero 'clean_data.parquet' guardado en la carpeta {data_folder}")